# **Restructured Resume Ranking System with BERT and XAI**

## 1. **Environment Setup and verification**

In [ ]:
# Check if CUDA is available for GPU acceleration
import torch

if torch.cuda.is_available():
    print("CUDA is available!")
    print("CUDA Version (from PyTorch):", torch.version.cuda)
else:
    print("CUDA is not available.")

Package Installation Instructions


In [ ]:
# Uncomment and run these lines if you need to install or update packages
# Deep Learning & Transformer packages
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# pip install transformers sentence-transformers

# XAI & Visualization packages
# pip install lime shap plotly pandas matplotlib seaborn

# Document processing packages
# pip install pdf2image pytesseract opencv-python pillow
# pip install fuzzywuzzy python-Levenshtein pdfplumber PyPDF2

# Text processing & NLP packages
# pip install python-docx spacy nltk scikit-learn

# Download required language models
# python -m spacy download en_core_web_sm
# python -m nltk.downloader punkt stopwords wordnet omw-1.4

Verify Transformer Functionality

In [ ]:
# Verify that PyTorch and Transformers are working correctly
import os
os.environ["TRANSFORMERS_FRAMEWORK"] = "pt"  # Force PyTorch

import torch
from transformers import AutoTokenizer, AutoModel

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Test basic transformer functionality
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModel.from_pretrained("distilbert-base-uncased")
print("Transformer model loaded successfully!")

Test PDF Processing


In [ ]:
# Test PDF processing capabilities with Poppler
import os
from pdf2image import convert_from_path
import logging

# Setup logging
logging.basicConfig(level=logging.INFO)

# Test PDF path with fallback options
pdf_paths_to_try = [
    "Datasets/sample cv/scanned cv.pdf",
    "Datasets/sample_cv.pdf",
    # Add more potential paths if needed
]

# Find a valid PDF for testing
test_pdf_path = None
for path in pdf_paths_to_try:
    if os.path.exists(path):
        test_pdf_path = path
        break

if test_pdf_path:
    # Try with explicit poppler path or let it auto-detect
    poppler_paths = [
        r'C:\Program Files\poppler\poppler-24.08.0\Library\bin',
        r'C:\Program Files\poppler\bin',
        None  # Let pdf2image try to find poppler automatically
    ]
    
    success = False
    for poppler_path in poppler_paths:
        try:
            if poppler_path:
                print(f"Testing PDF conversion with Poppler at: {poppler_path}")
                images = convert_from_path(test_pdf_path, poppler_path=poppler_path, first_page=1, last_page=1)
            else:
                print("Testing PDF conversion with auto-detected Poppler")
                images = convert_from_path(test_pdf_path, first_page=1, last_page=1)
                
            print(f"Success! Converted 1 page from PDF to image.")
            success = True
            break
        except Exception as e:
            print(f"Error with path {poppler_path}: {e}")
    
    if not success:
        print("Failed to convert PDF with any Poppler configuration. Please ensure Poppler is installed correctly.")
else:
    print("No sample PDF files found. Please provide a valid PDF path.")

## **2. Import Required Libraries**

In [ ]:
# Set environment variable to force PyTorch before importing transformers
import os
os.environ["TRANSFORMERS_FRAMEWORK"] = "pt"  # Tell transformers to use PyTorch only

# Standard imports
from tqdm.notebook import tqdm
import torch
from transformers import AutoTokenizer, AutoModel
import warnings
warnings.filterwarnings('ignore')

# Import pipeline with explicit framework setting
from transformers import pipeline as transformers_pipeline  # Renamed to avoid confusion

# Other required imports
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.decomposition import PCA

# Import LIME and SHAP with error handling
try:
    import lime
    import lime.lime_text
    has_lime = True
except ImportError:
    has_lime = False
    print("LIME not available - XAI features will be limited")
    
try:
    import shap
    has_shap = True
except ImportError:
    has_shap = False
    print("SHAP not available - XAI features will be limited")

## **3. Transformer Model Integration**

In [ ]:
class TransformerModelHandler:
    """Handler for transformer-based NLP models"""
    
    def __init__(self, model_name="distilbert-base-uncased", device=None):
        """Initialize transformer models
        
        Args:
            model_name: The pre-trained model to use
            device: 'cpu' or 'cuda' or None (auto-detect)
        """
        self.model_name = model_name
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.initialized = False
        
        print(f"Initializing {model_name} on {self.device}")
        
        # Initialize tokenizer and model with proper error handling
        try:
            # Initialize main model components first
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModel.from_pretrained(model_name).to(self.device)
            self.initialized = True
            print(f"Successfully initialized main {model_name} components")
            
            # Initialize optional components with separate try-except
            try:
                # Initialize Sentence Transformer for semantic similarity
                self.sentence_model = SentenceTransformer('all-MiniLM-L6-v2')
                print("SentenceTransformer initialized")
            except Exception as e:
                print(f"SentenceTransformer initialization error (optional): {str(e)}")
                self.sentence_model = None
            
            try:
                # Initialize NER pipeline with explicit PyTorch framework
                self.ner_pipeline = transformers_pipeline(
                    'ner',
                    model='dbmdz/bert-large-cased-finetuned-conll03-english',
                    tokenizer='dbmdz/bert-large-cased-finetuned-conll03-english',
                    device=0 if self.device == 'cuda' else -1,
                    framework="pt"  # Explicitly use PyTorch
                )
                print("NER pipeline initialized with PyTorch")
            except Exception as e:
                print(f"NER pipeline initialization error (optional): {str(e)}")
                self.ner_pipeline = None
                
        except Exception as e:
            self.initialized = False
            print(f"Error initializing transformer models: {str(e)}")
            print("Trying fallback model...")
            try:
                # Try a smaller model as fallback
                fallback_model = "distilbert-base-uncased"
                if model_name != fallback_model:
                    self.model_name = fallback_model
                    self.tokenizer = AutoTokenizer.from_pretrained(fallback_model)
                    self.model = AutoModel.from_pretrained(fallback_model).to(self.device)
                    self.initialized = True
                    print(f"Successfully initialized fallback model {fallback_model}")
                else:
                    print("Fallback model also failed")
            except Exception as e2:
                print(f"Fallback model initialization also failed: {str(e2)}")

### Text Embedding and Semantic Similarity Methods


In [ ]:
# Add these methods to the TransformerModelHandler class

def get_embeddings(self, text, pooling='mean'):
    """Get embeddings for input text
    
    Args:
        text: Text to embed
        pooling: Pooling strategy ('mean', 'cls', or 'max')
        
    Returns:
        Tensor containing embeddings
    """
    if not self.initialized:
        print("Model not properly initialized")
        return None
    
    # Tokenize and get model output
    inputs = self.tokenizer(
        text,
        return_tensors="pt", 
        padding=True, 
        truncation=True, 
        max_length=512
    ).to(self.device)
    
    with torch.no_grad():
        outputs = self.model(**inputs)
    
    # Apply pooling strategy
    if pooling == 'cls':
        embeddings = outputs.last_hidden_state[:, 0, :]
    elif pooling == 'mean':
        # Mean pooling - take attention mask into account for averaging
        attention_mask = inputs['attention_mask']
        token_embeddings = outputs.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    else:  # max pooling
        token_embeddings = outputs.last_hidden_state
        input_mask_expanded = inputs['attention_mask'].unsqueeze(-1).expand(token_embeddings.size()).float()
        token_embeddings[input_mask_expanded == 0] = -1e9  # Set padding tokens to large negative value
        embeddings = torch.max(token_embeddings, 1)[0]
    
    return embeddings

def get_semantic_similarity(self, text1, text2):
    """Calculate semantic similarity between two texts using SentenceTransformer
    
    Args:
        text1: First text
        text2: Second text
        
    Returns:
        Similarity score between 0 and 1
    """
    if not hasattr(self, 'sentence_model') or self.sentence_model is None:
        print("Sentence transformer model not available")
        return 0.0
        
    try:
        embedding1 = self.sentence_model.encode(text1, convert_to_tensor=True)
        embedding2 = self.sentence_model.encode(text2, convert_to_tensor=True)
        
        # Calculate cosine similarity and convert to numpy (if needed)
        cos_sim = torch.nn.functional.cosine_similarity(embedding1.unsqueeze(0), 
                                                       embedding2.unsqueeze(0))
        return cos_sim.item()
    except Exception as e:
        print(f"Error computing semantic similarity: {str(e)}")
        return 0.0

### Named Entity Recognition Method


In [ ]:
# Add this method to the TransformerModelHandler class

def extract_entities(self, text):
    """Extract named entities from text using a transformer model
    
    Args:
        text: Input text
        
    Returns:
        Dictionary of entities by type
    """
    if not hasattr(self, 'ner_pipeline') or self.ner_pipeline is None:
        print("NER pipeline not available")
        return {}
    
    try:
        # Process text in chunks to handle long documents
        max_length = 512
        chunks = [text[i:i+max_length] for i in range(0, len(text), max_length)]
        
        all_entities = []
        for chunk in chunks:
            try:
                entities = self.ner_pipeline(chunk)
                all_entities.extend(entities)
            except Exception as chunk_error:
                print(f"Error processing chunk: {str(chunk_error)}")
                continue
        
        # Group by entity type
        grouped_entities = {}
        for entity in all_entities:
            entity_type = entity['entity']
            if entity_type.startswith('B-') or entity_type.startswith('I-'):
                entity_type = entity_type[2:]  # Remove B- or I- prefix
            
            if entity_type not in grouped_entities:
                grouped_entities[entity_type] = []
            
            grouped_entities[entity_type].append(entity['word'])
        
        # Remove duplicates
        for entity_type in grouped_entities:
            grouped_entities[entity_type] = list(set(grouped_entities[entity_type]))
        
        return grouped_entities
    
    except Exception as e:
        print(f"Error in entity extraction: {str(e)}")
        return {}

# Initialize transformer handler as a standalone utility
transformer_handler = TransformerModelHandler(model_name="distilbert-base-uncased")

## **4. Explainability AI (XAI) Integration**

In [ ]:
class ExplainabilityEngine:
    """Provides explainable AI capabilities for resume ranking decisions"""
    
    def __init__(self):
        """Initialize XAI components"""
        self.explainers = {}
        self.feature_importance = {}
    
    def create_lime_explainer(self, prediction_fn, class_names, feature_names=None):
        """Create a LIME explainer for text data"""
        try:
            from lime.lime_text import LimeTextExplainer
            
            # Wrap prediction function to ensure it handles lists properly
            def wrapped_prediction(texts):
                # LIME expects a 2D array for classification
                result = prediction_fn(texts)
                if len(result.shape) == 1:
                    # If result is 1D for a single sample, reshape to 2D
                    return result.reshape(1, -1)
                return result
            
            # Create the explainer with the wrapped prediction function
            self.explainers['lime'] = LimeTextExplainer(class_names=class_names)
            # Store the wrapped function
            self.wrapped_prediction_fn = wrapped_prediction
            return True
        except Exception as e:
            print(f"Error creating LIME explainer: {str(e)}")
            return False

### LIME Explanation Method


In [ ]:
df.iloc[0,14]

## to see job seeker feedback

In [ ]:
# Add this after main execution
if os.path.exists("candidate_feedback"):
    sample_file = next(os.walk("candidate_feedback"))[2][0]
    with open(os.path.join("candidate_feedback", sample_file), 'r') as f:
        print("\nSample Feedback Content:")
        print(f.read())

In [ ]:
df.head(20)